# 03. 단일 라벨 기준 모델 (비교용 baseline)

**목적**: 14주차 발표의 *"기존 모델(단일 라벨) vs 개선 모델(다중 라벨)"* 비교 자료 만들기.

| | 기존 방식 (이 노트북) | 개선 방식 (02 노트북) |
|---|---|---|
| 출력 | 44개 중 **감정 1개** | 44개 감정 **각각 0~1** |
| 활성화 / 손실 | Softmax + `CrossEntropyLoss` | Sigmoid + `BCEWithLogitsLoss` |
| 예측 | argmax 1개 | threshold 넘는 감정 전부 |

- KOTE는 원래 문장마다 감정이 여러 개 → 단일 라벨 학습용으로 **정답 중 1개를 무작위(seed 고정)로 골라** 사용.
  (첫 번째 라벨을 쓰면 번호가 작은 '불평/불만' 쪽으로 쏠려서 무작위로 함)
- **공정한 비교**를 위해 두 모델을 같은 기준으로 잼:
  - **Hit@1**: 1위로 예측한 감정이 정답 감정들 안에 있는 비율
  - **Hit@3**: 상위 3개 중 하나라도 정답 안에 있는 비율
  - **F1 (top-1)**: 감정 1개만 예측했다고 보고 전체 정답과 비교한 micro/macro F1
  - **F1 (threshold)**: 다중 라벨 모델만 가능 (감정 여러 개 예측)
- 마지막 셀에서 `models/`의 다중 라벨 모델(02에서 저장한 것)을 불러와 비교표를 만들고 `outputs/`에 csv로 저장.

**실행**: 커널 `kobert (3.11)` 선택 → **Run All**.  
⚠️ 비교가 공정하려면 셀 2의 `TRAIN_N`, `EPOCHS`를 **02에서 학습한 값과 똑같이** 맞출 것.

In [ ]:
# 1. 기본 설정
import sys, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

import transformers
print("python      :", sys.executable)
print("transformers:", transformers.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device      :", device)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

project_root = Path.cwd().parent
raw_dir    = project_root / "data" / "raw"
model_dir  = project_root / "models"
output_dir = project_root / "outputs"
model_dir.mkdir(exist_ok=True); output_dir.mkdir(exist_ok=True)
print("raw 폴더    :", raw_dir, raw_dir.exists())

In [ ]:
# 2. 하이퍼파라미터 — 02 노트북과 똑같이 맞추기!
TRAIN_N    = 2000    # 02를 전체로 학습했다면 None
VAL_N      = 1000    # 02를 전체로 학습했다면 None
MAX_LENGTH = 64
BATCH_SIZE = 16
EPOCHS     = 2       # 02와 같은 값
LR         = 2e-5
NUM_LABELS = 44

In [ ]:
# 3. 감정 이름 (KOTE 라벨 순서)
LABELS = ['불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경','기대감','우쭐댐/무시함',
          '안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심','아껴주는','부끄러움','공포/무서움','절망',
          '한심함','역겨움/징그러움','짜증','어이없음','없음','패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음',
          '죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처','경악','부담/안 내킴','서러움','재미없음','불쌍함/연민','놀람',
          '행복','불안/걱정','기쁨','안심/신뢰']
print("레이블 수:", len(LABELS))

In [ ]:
# 4. 데이터 불러오기 + 단일 라벨 만들기
rng = np.random.default_rng(SEED)

def load_kote(name):
    df = pd.read_csv(raw_dir / f"{name}.tsv", sep="\t", header=None, names=["id", "text", "labels"])
    df["text"] = df["text"].astype(str)
    df["label_list"] = df["labels"].apply(lambda s: [int(x) for x in str(s).split(",")])
    df["multi_hot"] = df["label_list"].apply(lambda ls: [1 if i in ls else 0 for i in range(NUM_LABELS)])
    df["single"] = df["label_list"].apply(lambda ls: int(rng.choice(ls)))   # 정답 중 1개 무작위
    return df

train_df = load_kote("train")
val_df   = load_kote("val")
test_df  = load_kote("test")

if TRAIN_N: train_df = train_df.sample(n=TRAIN_N, random_state=SEED)
if VAL_N:   val_df   = val_df.sample(n=VAL_N, random_state=SEED)
print("사용할 데이터 → train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

print("\n예시 (원래 정답 여러 개 → 단일 라벨 1개):")
for _, r in train_df.head(3).iterrows():
    print(f"- {r.text[:40]}...\n    전체: {[LABELS[i] for i in r.label_list]}\n    단일: {LABELS[r.single]}")

print("\n단일 라벨 분포 Top 10:")
print(train_df["single"].map(lambda i: LABELS[i]).value_counts().head(10).to_string())

In [ ]:
# 5. KoBERT 토크나이저 + 정상 동작 확인
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel

MODEL_NAME = "skt/kobert-base-v1"
tokenizer = KoBERTTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.encode("한국어 모델을 공유합니다.") == [2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3], \
    "토크나이저 이상! transformers 4.44.2 + kobert_tokenizer 확인"
print("토크나이저 정상 ✅")

In [ ]:
# 6. Dataset / DataLoader — 정답은 숫자 1개(single), 평가용으로 multi_hot도 같이
class KOTEDataset(Dataset):
    def __init__(self, df):
        self.texts  = df["text"].tolist()
        self.single = df["single"].tolist()
        self.multi  = df["multi_hot"].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.single[idx], self.multi[idx]

def collate(batch):
    texts, single, multi = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc["labels"]    = torch.tensor(single, dtype=torch.long)      # CrossEntropy는 정수 정답
    enc["multi_hot"] = torch.tensor(multi,  dtype=torch.float32)
    return enc

train_loader = DataLoader(KOTEDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(KOTEDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(KOTEDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

b = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in b.items()})

In [ ]:
# 7. 모델: 구조는 02와 완전히 같음 (KoBERT → [CLS] → Linear 768→44)
#    다른 건 손실함수뿐: CrossEntropyLoss = Softmax + 정답 1개
class KoBERTClassifier(nn.Module):
    def __init__(self, num_labels=NUM_LABELS, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        return self.classifier(self.dropout(out.last_hidden_state[:, 0, :]))   # logits (B, 44)

model = KoBERTClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [ ]:
# 8. 예측 / 평가 함수 (두 모델 공통으로 씀)
def run_inputs(batch):
    return dict(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                token_type_ids=batch.get("token_type_ids"))

@torch.no_grad()
def predict(m, loader, act="softmax"):
    m.eval()
    P, M, S = [], [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = m(**run_inputs(batch))
        P.append((torch.softmax(logits, -1) if act == "softmax" else torch.sigmoid(logits)).cpu().numpy())
        M.append(batch["multi_hot"].cpu().numpy())
        S.append(batch["labels"].cpu().numpy())
    return np.concatenate(P), np.concatenate(M), np.concatenate(S)

def top1_metrics(probs, multi, single):
    top1 = probs.argmax(1)
    top3 = probs.argsort(1)[:, ::-1][:, :3]
    idx = np.arange(len(probs))
    one_hot = np.zeros_like(multi); one_hot[idx, top1] = 1      # "감정 1개만 예측"
    return {
        "Hit@1":           (multi[idx, top1] == 1).mean(),
        "Hit@3":           (multi[idx[:, None], top3].max(1) == 1).mean(),
        "정확도(뽑은 1개와 일치)": (top1 == single).mean(),
        "micro-F1 (top-1)": f1_score(multi, one_hot, average="micro", zero_division=0),
        "macro-F1 (top-1)": f1_score(multi, one_hot, average="macro", zero_division=0),
    }

def show(d):
    return " | ".join(f"{k} {v:.4f}" for k, v in d.items())

In [ ]:
# 9. 학습
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = criterion(model(**run_inputs(batch)), batch["labels"])
        loss.backward()
        optimizer.step()
        total += loss.item()
        if step % 100 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {loss.item():.4f}")

    vp, vm, vs = predict(model, val_loader, "softmax")
    print(f"[Epoch {epoch}] train loss {total/len(train_loader):.4f} | {(time.time()-start)/60:.1f}분")
    print("   val →", show(top1_metrics(vp, vm, vs)))

In [ ]:
# 10. 테스트셋 성능 (단일 라벨 모델)
sp, sm, ss = predict(model, test_loader, "softmax")
single_res = top1_metrics(sp, sm, ss)
print("단일 라벨 TEST →", show(single_res))

save_path = model_dir / f"kobert_kote_single_n{TRAIN_N or 'all'}_e{EPOCHS}.pt"
torch.save({"state_dict": model.state_dict(), "labels": LABELS, "max_length": MAX_LENGTH}, save_path)
print("저장:", save_path)

In [ ]:
# 11. 다중 라벨 모델(02에서 저장) 불러와서 같은 기준으로 평가
#     구조가 같아서 KoBERTClassifier에 그대로 로드 가능
multi_path = model_dir / f"kobert_kote_n{TRAIN_N or 'all'}_e{EPOCHS}.pt"   # 같은 설정으로 학습한 파일
if not multi_path.exists():
    cands = sorted(model_dir.glob("kobert_kote_n*.pt"), key=lambda p: p.stat().st_mtime)
    assert cands, "models/에 다중 라벨 모델이 없음 → 02 노트북을 먼저 실행"
    multi_path = cands[-1]
    print("⚠️ 같은 설정의 파일이 없어 가장 최근 파일 사용 (설정이 다르면 비교가 불공정할 수 있음)")
print("다중 라벨 모델:", multi_path.name)

ckpt = torch.load(multi_path, map_location=device, weights_only=False)
multi_model = KoBERTClassifier().to(device)
multi_model.load_state_dict(ckpt["state_dict"])
th = ckpt.get("threshold", 0.3)

mp, mm, ms = predict(multi_model, test_loader, "sigmoid")
multi_res = top1_metrics(mp, mm, ms)
pred_th = (mp >= th).astype(int)
multi_res["micro-F1 (threshold)"] = f1_score(mm, pred_th, average="micro", zero_division=0)
multi_res["macro-F1 (threshold)"] = f1_score(mm, pred_th, average="macro", zero_division=0)
print(f"다중 라벨 TEST (th={th}) →", show(multi_res))
print("평균 예측 감정 수:", round(pred_th.sum(1).mean(), 2), "/ 실제 평균:", round(mm.sum(1).mean(), 2))

In [ ]:
# 12. 비교표 (발표용) → outputs/ 에 csv 저장
table = pd.DataFrame({"단일 라벨 (Softmax+CE)": single_res,
                      "다중 라벨 (Sigmoid+BCE)": multi_res}).round(4)
table.loc["예측 감정 수"] = ["항상 1개", f"평균 {pred_th.sum(1).mean():.2f}개"]
display(table)

csv_path = output_dir / f"compare_single_vs_multi_n{TRAIN_N or 'all'}_e{EPOCHS}.csv"
table.to_csv(csv_path, encoding="utf-8-sig")
print("저장:", csv_path)

In [ ]:
# 13. 같은 문장, 두 모델 예측 비교 (발표 캡처용)
@torch.no_grad()
def compare(text, k=3):
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    enc = {k_: v.to(device) for k_, v in enc.items()}
    model.eval(); multi_model.eval()
    p1 = torch.softmax(model(**run_inputs(enc)), -1)[0].cpu().numpy()
    p2 = torch.sigmoid(multi_model(**run_inputs(enc)))[0].cpu().numpy()
    print("\n문장:", text)
    print("  단일 라벨 →", LABELS[p1.argmax()], f"({p1.max():.2f})")
    picked = [f"{LABELS[i]} {p2[i]:.2f}" for i in p2.argsort()[::-1] if p2[i] >= th]
    print(f"  다중 라벨 → " + (", ".join(picked) or "없음") + f"  (th={th})")

for s in ["오늘 시험을 잘 봐서 뿌듯하다.",
          "다음 발표가 걱정된다.",
          "오늘 시험을 잘 봐서 뿌듯했지만 다음 발표가 걱정되고 너무 피곤하다."]:
    compare(s)